# <font color = 'indianred'>**Multilabel Classification of StackExchange Dataset using GEMMA** </font>

**Objective:**

In this notebook, we aim to use GEMMA models with QLORA for classification problems. **We will now use Casual Languagge Model - Basically we will do instruction tuning.**


**Plan**

1. Set Environment
2. Load Dataset
3. Accessing and Manipulating Splits
4. Load Pre-trained Tokenizer
5. Create Prompts
6. Model Training
1. Download pre-trained model <br>  
3. PEFT Setup
4. Training Arguments <br>
5. Instantiate Trainer <br>
6. Setup WandB <br>
7. Training























# <font color = 'indianred'> **1. Setting up the Environment** </font>



In [ ]:
!nvidia-smi

Sat Dec 13 19:49:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
# If in Colab, then import the drive module from google.colab
if 'google.colab' in str(get_ipython()):
  # !pip install uv
  !pip install numpy -U -qq
  !pip install transformers evaluate wandb datasets accelerate trl peft bitsandbytes -U -qq
  !pip uninstall tensorflow -y -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 121.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.5 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

 <Font size = 5 color = 'indianred'>**Restart the session before moving onto next cell**
> Runtime- Restart Session

<font color = 'indianred'> *Load Libraries* </font>

In [1]:
# standard python libraries
from pathlib import Path
from typing import Dict, List, Union, Optional, Tuple
from tqdm import tqdm
import json
import joblib
import os
import sys

# Data Science librraies
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import multilabel_confusion_matrix, precision_score, recall_score, f1_score

# Pytorch
import torch
import torch.nn as nn

# Huggingface Librraies
import evaluate
from datasets import load_dataset, DatasetDict, Dataset, ClassLabel
from trl import SFTConfig, SFTTrainer
from transformers import (
    set_seed,
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    AutoConfig,
    BitsAndBytesConfig,
)
from peft import (
    TaskType,
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
    AutoPeftModelForCausalLM,
    PeftConfig
)
# Logging and secrets
from huggingface_hub import login, HfApi, create_repo
from google.colab import userdata
import wandb



In [2]:
sys.path

['/content',
 '/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.12/dist-packages/IPython/extensions',
 '/root/.ipython',
 '/tmp/tmp9v_iod7b']

In [3]:
# If running on Google Colab, use Google Drive as storage
# CHANGE FOLDERS TO WHERE YOU WANT TO SAVE DATA AND MODELS

if 'google.colab' in str(get_ipython()):
    from google.colab import drive  # Import Google Drive mounting utility
    drive.mount('/content/drive')  # Mount Google Drive

    # Set base folder path for storing data on Google Drive
    data_folder= Path('/content/drive/MyDrive')
    project_folder = Path('/content/drive/MyDrive')
    base_folder = Path('/content/drive/MyDrive')

# If running locally, specify a different path
else:
    # Set base folder path for storing data on local machine
    data_folder= Path('/home/harpreet/Insync/google_drive_shaannoor/data')
    project_folder= Path('/home/harpreet/Insync/google_drive_shaannoor/teaching_fall_2025/LLM_Fall_2025/')

Mounted at /content/drive


In [4]:
util_folder = project_folder/'shared_utils'
sys.path.append(str(util_folder))

In [5]:
from shared_utils import free_gpu_memory, find_linear_layers, multilabel_evaluation, get_appropriate_dtype

In [6]:
wandb_api_key = userdata.get('WANDB_API_KEY')
hf_token = userdata.get('HF_TOKEN')

In [7]:
if hf_token:
    # Log in to Hugging Face
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Hugging Face token not found in notebook secrets.")

Successfully logged in to Hugging Face!


In [8]:
if wandb_api_key:
  wandb.login(key=wandb_api_key)
  print("Successfully logged in to WANDB!")
else:
    print("WANDB key not found in notebook secrets.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: younes-hoseini67 (younes-hoseini67-university-of-texas-at-dallas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged in to WANDB!


In [9]:
kaggle_api = base_folder/'.kaggle'
import os
if 'google.colab' in str(get_ipython()):
    os.environ['KAGGLE_CONFIG_DIR'] = "/content/drive/MyDrive/.kaggle/"
if 'google.colab' in str(get_ipython()):
    !chmod 600 /content/drive/MyDrive/.kaggle/kaggle.json
if 'google.colab' in str(get_ipython()):
    ! ls -la  /content/drive/MyDrive/.kaggle/kaggle.json

-rw------- 1 root root 67 Oct 27 23:03 /content/drive/MyDrive/.kaggle/kaggle.json


# <font color = 'indianred'> **2. Load Data set**
    


In [10]:
!kaggle competitions download -c emotion-detection-fall-2025-dl
!unzip -o emotion-detection-fall-2025-dl.zip -d {data_folder}
train = pd.read_csv(data_folder/'train.csv')
print(f"Kaggle train data shape: {train.shape}")
print(f"Columns: {train.columns.tolist()}")

# Use the Kaggle training data
df = train
df

  0% 0.00/609k [00:00<?, ?B/s]
100% 609k/609k [00:00<00:00, 1.19GB/s]
Archive:  emotion-detection-fall-2025-dl.zip
  inflating: /content/drive/MyDrive/sample_submission.csv  
  inflating: /content/drive/MyDrive/test.csv  
  inflating: /content/drive/MyDrive/train.csv  
Kaggle train data shape: (7724, 13)
Columns: ['ID', 'Tweet', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']


,ID,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,2017-21441,“Worry is a down payment on a problem you may ...,0,1,0,0,0,0,1,0,0,0,1
1,2017-31535,Whatever you decide to do make sure it makes y...,0,0,0,0,1,1,1,0,0,0,0
2,2017-21068,@Max_Kellerman it also helps that the majorit...,1,0,1,0,1,0,1,0,0,0,0
3,2017-31436,Accept the challenges so that you can literall...,0,0,0,0,1,0,1,0,0,0,0
4,2017-22195,My roommate: it's okay that we can't spell bec...,1,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7719,2018-01993,@BadHombreNPS @SecretaryPerry If this didn't m...,1,0,1,0,0,0,0,0,0,0,0
7720,2018-01784,Excited to watch #stateoforigin tonight! Come ...,0,0,0,0,1,0,1,0,0,0,0
7721,2018-04047,"Blah blah blah Kyrie, IT, etc. @CJC9BOSS leavi...",1,0,1,0,0,0,0,0,1,0,0
7722,2018-03041,#ThingsIveLearned The wise #shepherd never tru...,0,0,0,0,0,0,0,0,0,0,0


In [11]:
# Define emotion class names
class_names = ['anger', 'anticipation', 'disgust', 'fear', 'joy',
               'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']

# Convert binary columns to list of emotion labels
def get_emotion_labels(row):
    emotions = [emotion for emotion in class_names if row[emotion] == 1]
    return json.dumps(emotions)

df['label'] = df.apply(get_emotion_labels, axis=1)

# Create final dataframe with text and label
df_final = df[['Tweet', 'label']].rename(columns={'Tweet': 'text'})

print(f"\nFinal dataset shape: {df_final.shape}")
print(f"Sample label: {df_final['label'][0]}")

emotion_dataset_final = Dataset.from_pandas(df_final)
print(f"First example label: {emotion_dataset_final[0]['label']}")


Final dataset shape: (7724, 2)
Sample label: ["anticipation", "optimism", "trust"]
First example label: ["anticipation", "optimism", "trust"]


# <font color = 'indianred'> **3. Accessing and Manuplating Splits**</font>



<font color = 'indianred'>*Create futher subdivions of the splits*</font>

In [12]:
# Split the test set into test and validation sets
test_val_splits = emotion_dataset_final.train_test_split(test_size=0.4, seed=42)
train_split= test_val_splits['train']
test_val_splits = test_val_splits['test'].train_test_split(test_size=0.5, seed=42,)
val_split = test_val_splits['train']
test_split = test_val_splits['test']


<font color = 'indianred'>*small subset for initial experimenttaion*</font>

In [13]:
train_split = train_split.shuffle(seed = 42).select(range(min(2000, len(train_split))))
val_split = val_split.shuffle(seed = 42).select(range(min(2000, len(val_split))))
test_split = test_split.shuffle(seed = 42).select(range(min(2000, len(test_split))))

In [14]:
data_subset = DatasetDict({"train": train_split, "valid": val_split, "test": test_split})

In [15]:
data_subset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    valid: Dataset({
        features: ['text', 'label'],
        num_rows: 1545
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1545
    })
})

In [16]:
data_subset['train'][0]

{'text': 'Dear future big brother players, just chase dick all season and you too can win 500,00 dollars and an Sti #bb18 ',
 'label': '["anger", "joy", "optimism"]'}

# <font color = 'indianred'>**4. Load pre-trained Tokenizer**</font>

In [17]:
free_gpu_memory()

GPU memory has been freed.


In [18]:
checkpoint = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [19]:
tokenizer.eos_token

'<eos>'

In [20]:
tokenizer.pad_token

'<pad>'

In [21]:
tokenizer.padding_side

'left'

In [22]:
from pprint import pprint
print(tokenizer.chat_template)

{{ bos_token }}{% if messages[0]['role'] == 'system' %}{{ raise_exception('System role not supported') }}{% endif %}{% for message in messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}{% endif %}{% if (message['role'] == 'assistant') %}{% set role = 'model' %}{% else %}{% set role = message['role'] %}{% endif %}{{ '<start_of_turn>' + role + '
' + message['content'] | trim + '<end_of_turn>
' }}{% endfor %}{% if add_generation_prompt %}{{'<start_of_turn>model
'}}{% endif %}


In [23]:
# 2. Test the template works
print("Testing custom chat template...")
test_result = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Test"}, {"role": "assistant", "content": "Response"}],
    return_dict=True,
    return_assistant_tokens_mask=True,
    add_generation_prompt=False
)
print(test_result)
assistant_tokens = sum(test_result.get('assistant_masks', []))
print(f"Assistant tokens detected: {assistant_tokens}")

return_assistant_tokens_mask==True but chat template does not contain `{% generation %}` keyword.


Testing custom chat template...
{'input_ids': [2, 106, 1645, 108, 2015, 107, 108, 106, 2516, 108, 3943, 107, 108], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'assistant_masks': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}
Assistant tokens detected: 0


In [24]:
# Modified template to only train on the actual content
tokenizer.chat_template = """
{{ bos_token }}
{% if messages[0]['role'] == 'system' %}
  {{ raise_exception('System role not supported') }}
{% endif %}
{% for message in messages %}
  {% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}
    {{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}
  {% endif %}
  {% if message['role'] == 'assistant' %}
<start_of_turn>model
{% generation %}{{ message['content'] | trim }}
<end_of_turn>{% endgeneration %}
  {% else %}
<start_of_turn>{{ message['role'] }}
{{ message['content'] | trim }}
<end_of_turn>
  {% endif %}
{% endfor %}
{% if add_generation_prompt %}
<start_of_turn>model
{% endif %}
"""


In [25]:
# 2. Test the template works
print("Testing custom chat template...")
test_result = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Test"}, {"role": "assistant", "content": "Response"}],
    return_dict=True,
    return_assistant_tokens_mask=True,
    add_generation_prompt=False
)
print(test_result)
assistant_tokens = sum(test_result.get('assistant_masks', []))
print(f"Assistant tokens detected: {assistant_tokens}")

Testing custom chat template...
{'input_ids': [108, 2, 108, 106, 1645, 108, 2015, 108, 107, 108, 106, 2516, 108, 3943, 108, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'assistant_masks': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1]}
Assistant tokens detected: 3


In [26]:
# Decode individual tokens to see what they are
print("\nToken-by-token breakdown:")
print("Index | Token ID | Token Text | Assistant?")
print("-" * 50)
input_ids = test_result['input_ids']
assistant_masks = test_result['assistant_masks']

for i, (token_id, is_assistant) in enumerate(zip(input_ids, assistant_masks)):
    token_text = tokenizer.decode([token_id])
    status = "ASSISTANT" if is_assistant else "Other"
    print(f"{i:5d} | {token_id:8d} | {repr(token_text):15s} | {status}")

print(f"\nFull text: {repr(tokenizer.decode(input_ids))}")

# Show only assistant tokens
assistant_token_ids = [token_id for token_id, is_assistant in zip(input_ids, assistant_masks) if is_assistant]
assistant_only_text = tokenizer.decode(assistant_token_ids)
print(f"Assistant-only text: {repr(assistant_only_text)}")

# Show the masked (non-assistant) tokens
user_token_ids = [token_id for token_id, is_assistant in zip(input_ids, assistant_masks) if not is_assistant]
user_only_text = tokenizer.decode(user_token_ids)
print(f"User/Other text: {repr(user_only_text)}")


Token-by-token breakdown:
Index | Token ID | Token Text | Assistant?
--------------------------------------------------
    0 |      108 | '\n'            | Other
    1 |        2 | '<bos>'         | Other
    2 |      108 | '\n'            | Other
    3 |      106 | '<start_of_turn>' | Other
    4 |     1645 | 'user'          | Other
    5 |      108 | '\n'            | Other
    6 |     2015 | 'Test'          | Other
    7 |      108 | '\n'            | Other
    8 |      107 | '<end_of_turn>' | Other
    9 |      108 | '\n'            | Other
   10 |      106 | '<start_of_turn>' | Other
   11 |     2516 | 'model'         | Other
   12 |      108 | '\n'            | Other
   13 |     3943 | 'Response'      | ASSISTANT
   14 |      108 | '\n'            | ASSISTANT
   15 |      107 | '<end_of_turn>' | ASSISTANT

Full text: '\n<bos>\n<start_of_turn>user\nTest\n<end_of_turn>\n<start_of_turn>model\nResponse\n<end_of_turn>'
Assistant-only text: 'Response\n<end_of_turn>'
User/Other text: 

#<font color = 'indianred'> **5. Create Chat Dataset**



In [27]:
class_names

['anger',
 'anticipation',
 'disgust',
 'fear',
 'joy',
 'love',
 'optimism',
 'pessimism',
 'sadness',
 'surprise',
 'trust']

In [28]:
def format_chat(example):
    instruction = f"Classify the TEXT by selecting all applicable labels from the following list: {class_names}.\n\nTEXT: {example['text']}"
    messages = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": f"{example['label']}"}
    ]
    return {"messages": messages}


In [29]:
data_subset_chat = data_subset.map(format_chat, remove_columns=["text", "label"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1545 [00:00<?, ? examples/s]

Map:   0%|          | 0/1545 [00:00<?, ? examples/s]

In [30]:
data_subset_chat

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2000
    })
    valid: Dataset({
        features: ['messages'],
        num_rows: 1545
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 1545
    })
})

In [31]:
data_subset_chat['train'][0]

{'messages': [{'content': "Classify the TEXT by selecting all applicable labels from the following list: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust'].\n\nTEXT: Dear future big brother players, just chase dick all season and you too can win 500,00 dollars and an Sti #bb18 ",
   'role': 'user'},
  {'content': '["anger", "joy", "optimism"]', 'role': 'assistant'}]}

##  <font color = 'indianred'> **5.1 Filter Longer sequences**

In [32]:
def check_length(example):
    # Find the first user message
    text_to_check = None
    for msg in example.get("messages", []):
        if msg.get("role") == "user":
            text_to_check = msg.get("content", "").strip()
            break

    # Return False if there's no user message or empty text
    if not text_to_check:
        return False

    # Tokenize the text and check length
    encoding = tokenizer.encode(text_to_check)
    return len(encoding) <= 500


In [33]:
filtered_data_subset_chat = DatasetDict({
    split: data_subset_chat[split].filter(check_length)
    for split in data_subset_chat.keys()
})

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1545 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1545 [00:00<?, ? examples/s]

In [34]:
filtered_data_subset_chat

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2000
    })
    valid: Dataset({
        features: ['messages'],
        num_rows: 1545
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 1545
    })
})

In [35]:
filtered_data_subset_chat['train'][0]

{'messages': [{'content': "Classify the TEXT by selecting all applicable labels from the following list: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust'].\n\nTEXT: Dear future big brother players, just chase dick all season and you too can win 500,00 dollars and an Sti #bb18 ",
   'role': 'user'},
  {'content': '["anger", "joy", "optimism"]', 'role': 'assistant'}]}

##  <font color = 'indianred'> **5.2 Push Dataset to Hub**

In [36]:
filtered_data_subset_chat.push_to_hub(
    "Sayedyounes/stack_multilabel_subset_chat",
    private=False  # Set to True if you want it private
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  193kB /  193kB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  151kB /  151kB            

README.md:   0%|          | 0.00/544 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/sayedyounes/stack_multilabel_subset_chat/commit/ab89cdeb1522361f9e8d3b936359dfabb43bbdea', commit_message='Upload dataset', commit_description='', oid='ab89cdeb1522361f9e8d3b936359dfabb43bbdea', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sayedyounes/stack_multilabel_subset_chat', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sayedyounes/stack_multilabel_subset_chat'), pr_revision=None, pr_num=None)

#  <font color = 'indianred'> **6. Model Training**

##  <font color = 'indianred'> **6.1 Download pre-trained model**

In [37]:
def get_appropriate_dtype():
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0) >= (8, 0):
        return torch.bfloat16
    return torch.float16

In [38]:
torch_data_type = get_appropriate_dtype()
torch_data_type

torch.bfloat16

In [39]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True,
  bnb_4bit_compute_dtype=torch_data_type,
  bnb_4bit_quant_storage=torch_data_type,
)

In [40]:
model = AutoModelForCausalLM.from_pretrained(checkpoint,
                                             quantization_config=bnb_config,
                                             torch_dtype=torch_data_type,
                                             trust_remote_code=True,)


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

##  <font color = 'indianred'> **6.2 PEFT Setup**

In [41]:
model

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedfor

In [42]:
find_linear_layers(model)

['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'lm_head']


['o_proj',
 'up_proj',
 'lm_head',
 'gate_proj',
 'k_proj',
 'v_proj',
 'down_proj',
 'q_proj']

In [43]:
TaskType.CAUSAL_LM

<TaskType.CAUSAL_LM: 'CAUSAL_LM'>

In [44]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=128,
    lora_alpha=256,
    lora_dropout=0.01,
    target_modules = ['v_proj',  'q_proj',  'up_proj', 'o_proj', 'down_proj', 'gate_proj','k_proj'])



## <font color = 'indianred'> **6.3 Training Arguments**</font>







In [45]:
# Define the directory where model checkpoints will be saved
model_folder = Path("/content/models/gemma_qlora_lmh_inst")

# Create the directory if it doesn't exist
model_folder.mkdir(exist_ok=True, parents=True)
run_name= 'stack_exp_lmh_gemma_inst'

use_fp16 = torch_data_type == torch.float16
use_bf16 = torch_data_type == torch.bfloat16

# Configure training parameters
training_args = SFTConfig(
    seed = 42,
    dataset_text_field="text",
    max_length = 1024,
    packing = False,
    assistant_only_loss=True,

    # Training-specific configurations
    num_train_epochs=2,  # Total number of training epochs
    per_device_train_batch_size=16, # Number of samples per training batch for each device
    per_device_eval_batch_size=16,  # Number of samples per evaluation batch for each device
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant":False},
    # torch_empty_cache_steps=5,
    weight_decay=0.0,  # Apply L2 regularization to prevent overfitting
    learning_rate=1e-5,  # Step size for the optimizer during training
    optim='adamw_torch',  # Optimizer,

    # Checkpoint saving and model evaluation settings
    output_dir=str(model_folder),  # Directory to save model checkpoints
    eval_strategy='steps',  # Evaluate model at specified step intervals
    eval_steps=20,  # Perform evaluation every 10 training steps
    save_strategy="steps",  # Save model checkpoint at specified step intervals
    save_steps=20,  # Save a model checkpoint every 10 training steps
    load_best_model_at_end=True,  # Reload the best model at the end of training
    save_total_limit=2,  # Retain only the best and the most recent model checkpoints
    # Use 'accuracy' as the metric to determine the best model
    metric_for_best_model="eval_loss",
    greater_is_better=False,  # A model is 'better' if its accuracy is higher


    # Experiment logging configurations (commented out in this example)
    logging_strategy='steps',
    logging_steps=20,
    report_to='wandb',  # Log metrics and results to Weights & Biases platform
    run_name= run_name,  # Experiment name for Weights & Biases

    # Precision settings determined based on GPU capability
    fp16=use_fp16 ,  # Set True if torch_data_type is torch.float16
    bf16=use_bf16,  # Set True if torch_data_type is torch.bfloat16
    tf32=False,  # Disable tf32 unless you want to use Ampere specific optimization
)


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [46]:
# If gradient checkpointing is enabled, configure relevant settings
if training_args.gradient_checkpointing:
    model.config.use_cache = False  # Disable caching for compatibility



##  <font color = 'indianred'> **6.4 Initialize Trainer**</font>



In [47]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=filtered_data_subset_chat['train'],
    eval_dataset=filtered_data_subset_chat['valid'],
    peft_config=peft_config,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1545 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1545 [00:00<?, ? examples/s]

In [48]:
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))

In [49]:
batch

{'input_ids': tensor([[108,   2, 108,  ..., 108, 107,   0],
         [108,   2, 108,  ...,   0,   0,   0],
         [108,   2, 108,  ...,   0,   0,   0],
         ...,
         [108,   2, 108,  ...,   0,   0,   0],
         [108,   2, 108,  ...,   0,   0,   0],
         [108,   2, 108,  ...,   0,   0,   0]], device='cuda:0'),
 'labels': tensor([[-100, -100, -100,  ...,  108,  107, -100],
         [-100, -100, -100,  ..., -100, -100, -100],
         [-100, -100, -100,  ..., -100, -100, -100],
         ...,
         [-100, -100, -100,  ..., -100, -100, -100],
         [-100, -100, -100,  ..., -100, -100, -100],
         [-100, -100, -100,  ..., -100, -100, -100]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')}

In [50]:
print(batch['input_ids'][0][0:5])
print(tokenizer.decode(batch['input_ids'][0][0:5]))
print(batch['labels'][0][0:5])

tensor([ 108,    2,  108,  106, 1645], device='cuda:0')

<bos>
<start_of_turn>user
tensor([-100, -100, -100, -100, -100], device='cuda:0')


In [51]:
print(len(batch['input_ids'][0]))
print(len(batch['labels'][0]))

116
116


In [52]:
print(batch['input_ids'][0][0:5])
print(tokenizer.decode(batch['input_ids'][0][0:5]))
print(batch['labels'][0][0:5])

tensor([ 108,    2,  108,  106, 1645], device='cuda:0')

<bos>
<start_of_turn>user
tensor([-100, -100, -100, -100, -100], device='cuda:0')


In [53]:
print(f"\nINPUTS")
print(f"{'-'*80}")
print(batch['input_ids'][0][106:120])
print(f"\nLABELS")
print(f"{'-'*80}")
print(batch['labels'][0][106:120])
print(f"\nTokens")
print(f"{'-'*80}")
print(tokenizer.decode(batch['input_ids'][0][106:120]))


INPUTS
--------------------------------------------------------------------------------
tensor([  542, 66047,   824,   664, 37968,  1746,  4437,   108,   107,     0],
       device='cuda:0')

LABELS
--------------------------------------------------------------------------------
tensor([  542, 66047,   824,   664, 37968,  1746,  4437,   108,   107,  -100],
       device='cuda:0')

Tokens
--------------------------------------------------------------------------------
simism", "sadness"]
<end_of_turn><pad>


In [54]:
def verify_loss_masking(tokenizer, trainer, num_samples=3):
    """
    Verify which tokens contribute to loss (labels != -100)
    for a few samples from the training dataloader.
    """
    dataloader = trainer.get_train_dataloader()
    batch = next(iter(dataloader))

    for i in range(min(num_samples, len(batch["input_ids"]))):
        input_ids = batch["input_ids"][i]
        labels = batch["labels"][i]

        print(f"\n{'='*80}")
        print(f"Sample {i+1}")
        print(f"{'='*80}")

        # Decode full sequence for reference
        full_text = tokenizer.decode(input_ids, skip_special_tokens=False)
        print(f"\nFull text:\n{full_text}")

        # Identify tokens used for loss
        loss_token_indices = (labels != -100).nonzero(as_tuple=True)[0]

        if len(loss_token_indices) == 0:
            print("All tokens masked — no loss will be calculated.")
            continue

        print(f"\nTokens contributing to loss ({len(loss_token_indices)} total):")
        print(f"{'-'*80}")
        print(f"{'Index':<8} {'Token ID':<10} {'Token Text'}")
        print(f"{'-'*80}")

        for idx in loss_token_indices.tolist():
            token_id = input_ids[idx].item()
            token_text = tokenizer.decode([token_id], skip_special_tokens=False)
            print(f"{idx:<8} {token_id:<10} {repr(token_text)}")

        print(f"{'-'*80}")
        print(f"Percentage of tokens used for loss: {len(loss_token_indices)/len(labels)*100:.2f}%")

In [55]:
# Call this after creating your trainer
verify_loss_masking(tokenizer, trainer, num_samples=2)



Sample 1

Full text:

<bos>
<start_of_turn>user
Classify the TEXT by selecting all applicable labels from the following list: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust'].

TEXT: @LindaFrum sadly, at least 3 more years of this. At what point will even the low info selfie lovers who voted for them say enough is enough
<end_of_turn>
<start_of_turn>model
["disgust", "pessimism", "sadness"]
<end_of_turn><pad>

Tokens contributing to loss (15 total):
--------------------------------------------------------------------------------
Index    Token ID   Token Text
--------------------------------------------------------------------------------
100      3681       '["'
101      1999       'dis'
102      15819      'gust'
103      824        '",'
104      664        ' "'
105      3462       'pes'
106      542        'si'
107      66047      'mism'
108      824        '",'
109      664        ' "'
110      37968      'sad'
11

## <font color = 'indianred'> **6.5 Setup WandB**</font>

In [56]:
%env WANDB_PROJECT = emotion-detection

env: WANDB_PROJECT=emotion-detection


##  <font color = 'indianred'> **6.6 Training**

In [57]:
try:
    # Your code that may cause a CUDA out-of-memory error
    # Example: trainer.train() or other GPU intensive operations
    # lora_model.config.use_cache = False
    trainer.train()
except RuntimeError as e:
    if 'CUDA out of memory' in str(e):
        print("CUDA out of memory error detected. Freeing GPU memory.")
        free_gpu_memory()
        # Optionally, you can retry the operation here after freeing up memory
        # Example retry:
        # trainer.train()
    else:
        raise e


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,0.740700,0.350796,1.673610,65415.000000,0.869659
40,0.332800,0.341659,1.684716,130386.000000,0.870841
60,0.320600,0.321953,1.670494,196279.000000,0.876915
80,0.294100,0.314907,1.560170,259994.000000,0.881762
100,0.272000,0.310835,1.578227,325385.000000,0.882215
120,0.278700,0.305756,1.572467,390868.000000,0.883470


##  <font color = 'indianred'> **6.6 Push best checkpoint to Hub**

In [58]:
best_model_checkpoint_step = trainer.state.best_model_checkpoint.split('-')[-1]

In [59]:
best_model_checkpoint_step

'120'

In [60]:
checkpoint = str(model_folder/f'checkpoint-{best_model_checkpoint_step}')
checkpoint

'/content/models/gemma_qlora_lmh_inst/checkpoint-120'

In [62]:
# Step 1: Create the repository
repo_id="sayedyounes/stack_exc_multilabel_instr_lm_head"
create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=False,
    exist_ok=True
)

# Step 2: Upload the folder
api = HfApi()
api.upload_folder(
    folder_path=checkpoint,
    repo_id=repo_id,
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...point-120/tokenizer.model:   0%|          | 16.2kB / 4.24MB            

  ...adapter_model.safetensors:   0%|          | 17.1kB /  332MB            

  ...ckpoint-120/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...eckpoint-120/optimizer.pt:   0%|          | 90.9kB /  665MB            

  ...eckpoint-120/scheduler.pt: 100%|##########| 1.47kB / 1.47kB            

  ...kpoint-120/tokenizer.json: 100%|##########| 34.4MB / 34.4MB            

  ...int-120/training_args.bin:   7%|6         |   432B / 6.35kB            

CommitInfo(commit_url='https://huggingface.co/sayedyounes/stack_exc_multilabel_instr_lm_head/commit/ee8d6c6fa231420d29eb837aaaaf7da33d42660e', commit_message='Upload folder using huggingface_hub', commit_description='', oid='ee8d6c6fa231420d29eb837aaaaf7da33d42660e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sayedyounes/stack_exc_multilabel_instr_lm_head', endpoint='https://huggingface.co', repo_type='model', repo_id='sayedyounes/stack_exc_multilabel_instr_lm_head'), pr_revision=None, pr_num=None)

In [63]:
wandb.finish()

eval/entropy,▇█▇▁▂▂
eval/loss,█▇▄▂▂▁
eval/mean_token_accuracy,▁▂▅▇▇█
eval/num_tokens,▁▂▄▅▇█
eval/runtime,█▃▂▅▂▁
eval/samples_per_second,▁▆▇▄▇█
eval/steps_per_second,▁▆▇▄▇█
train/entropy,█▆█▃▁▁
train/epoch,▁▁▂▂▄▄▅▅▆▆███
train/global_step,▁▁▂▂▄▄▅▅▆▆███
+5,...


kaggle submission
